# Stock Price Prediction - Exploration Notebook

This notebook follows the same flow as the project code:

1. Load stock data
2. Explore closing prices
3. Split data in time order
4. Scale data without leakage
5. Create RNN sequences
6. Build the project model
7. Train with validation and early stopping
8. Evaluate actual vs predicted prices
9. Understand the train/validation gap

The goal is not to memorize code. The goal is to understand each step.

## 1. Setup

This cell makes the notebook work even though it is inside the `notebook` folder.

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd().parent

if PROJECT_ROOT.name != "stock-price-prediction":
    PROJECT_ROOT = Path.cwd()

SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from preprocessing import (
    create_sequences,
    create_test_sequences_with_context,
    get_close_prices,
    load_stock_data,
    scale_data,
    split_data,
)
from model import build_rnn_model

plt.style.use("default")

## 2. Project Settings

These values match the current app.

In [ ]:
DATA_PATH = "data/stock_data.csv"
MODEL_PATH = "models/stock_rnn.keras"
SCALER_PATH = "models/scaler.pkl"
HISTORY_PATH = "models/training_history.json"

SEQUENCE_LENGTH = 60
TRAIN_RATIO = 0.8
EPOCHS = 100
BATCH_SIZE = 32
RANDOM_SEED = 11

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

## 3. Load Data

The CSV file was downloaded from Yahoo Finance. It has multiple columns like Open, High, Low, Close, and Volume.

In [ ]:
data = load_stock_data(DATA_PATH)

data.head()

In [ ]:
data.info()

## 4. Select Closing Price

This app predicts the next closing price, so we only use the `Close` column.

In [ ]:
close_prices = get_close_prices(data)

print("Total prices:", len(close_prices))
print("First date:", close_prices.index.min())
print("Last date :", close_prices.index.max())

close_prices.head()

In [ ]:
close_prices.describe()

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(close_prices, label="Close Price")
plt.title("AAPL Closing Price")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 5. Train/Test Split

For time-series data, we do not shuffle rows. Older prices are used for training. Newer prices are used for testing.

In [ ]:
train_data, test_data = split_data(close_prices, train_ratio=TRAIN_RATIO)

print("Train start:", train_data.index.min())
print("Train end  :", train_data.index.max())
print("Test start :", test_data.index.min())
print("Test end   :", test_data.index.max())

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(train_data, label="Training Data")
plt.plot(test_data, label="Testing Data")
plt.title("Chronological Train/Test Split")
plt.xlabel("Date")
plt.ylabel("Close Price")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6. Scale Data Without Leakage

Important rule:

The scaler must learn only from training data. If the scaler learns from test data, that is data leakage.

In [ ]:
train_scaled, test_scaled, scaler = scale_data(train_data, test_data)

print("Train scaled shape:", train_scaled.shape)
print("Test scaled shape :", test_scaled.shape)
print("Scaler train min :", scaler.data_min_[0])
print("Scaler train max :", scaler.data_max_[0])
print("Test max price   :", test_data.max())

If the test max price is higher than the scaler train max, that is not leakage. It simply means the future test period has prices the model did not see during training.

## 7. Create RNN Sequences

The model reads 60 previous days and predicts the next day.

Example:

`day 1 to day 60 -> day 61`

In [ ]:
X_train, y_train = create_sequences(train_scaled, SEQUENCE_LENGTH)

X_test, y_test = create_test_sequences_with_context(
    train_scaled,
    test_scaled,
    SEQUENCE_LENGTH,
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

The test set uses the last 60 training prices only as past context. This is valid because those prices happened before the test period.

## 8. Build Model

The current model is a small residual RNN.

Instead of predicting the whole price from zero, it predicts a small change:

`predicted price = last known price + learned change`

This helps reduce smooth downward predictions and keeps the model beginner-friendly.

In [ ]:
tf.keras.backend.clear_session()
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

model = build_rnn_model(
    sequence_length=SEQUENCE_LENGTH,
    number_of_features=1,
)

model.summary()

## 9. Train Model

EarlyStopping watches validation loss. When validation loss stops improving, training stops and the best weights are restored.

In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=0.00001,
    ),
]

history = model.fit(
    X_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    shuffle=False,
    callbacks=callbacks,
    verbose=1,
)

## 10. Training vs Validation Loss

A small gap is normal. A large gap means overfitting. The goal is not always zero gap. The goal is low validation loss and reasonable test performance.

In [ ]:
training_loss = history.history["loss"]
validation_loss = history.history["val_loss"]

best_epoch_index = int(np.argmin(validation_loss))
best_epoch = best_epoch_index + 1
best_train_loss = training_loss[best_epoch_index]
best_val_loss = validation_loss[best_epoch_index]
loss_gap = best_val_loss - best_train_loss

print("Best epoch:", best_epoch)
print("Training loss at best epoch  :", best_train_loss)
print("Validation loss at best epoch:", best_val_loss)
print("Gap:", loss_gap)

epochs = range(1, len(training_loss) + 1)

plt.figure(figsize=(14, 6))
plt.plot(epochs, training_loss, label="Training Loss")
plt.plot(epochs, validation_loss, label="Validation Loss")
plt.scatter(best_epoch, best_val_loss, color="red", label=f"Best Validation Epoch: {best_epoch}")
plt.title(f"Training vs Validation Loss (Best Val: {best_val_loss:.6f}, Gap: {loss_gap:.6f})")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.ylim(bottom=0)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 11. Evaluate On Test Data

The test data is the future part of the dataset. This is the most important check after training.

In [ ]:
predictions = model.predict(X_test, verbose=0)

predicted_prices = scaler.inverse_transform(predictions)
actual_prices = scaler.inverse_transform(y_test.reshape(-1, 1))

mae = mean_absolute_error(actual_prices, predicted_prices)
rmse = np.sqrt(mean_squared_error(actual_prices, predicted_prices))
mape = np.mean(np.abs((actual_prices - predicted_prices) / actual_prices)) * 100

print(f"MAE  : ${mae:.2f}")
print(f"RMSE : ${rmse:.2f}")
print(f"MAPE : {mape:.2f}%")

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(actual_prices, label="Actual")
plt.plot(predicted_prices, label="Predicted")
plt.title("Actual vs Predicted Stock Price - Test Data")
plt.xlabel("Test Sequence")
plt.ylabel("Stock Price")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 12. Compare With Simple Baseline

A simple baseline predicts that tomorrow's price will be the same as today's price. Stock models should be compared with this because it is hard to beat.

In [ ]:
baseline_predictions = scaler.inverse_transform(X_test[:, -1, :])

baseline_mae = mean_absolute_error(actual_prices, baseline_predictions)
baseline_rmse = np.sqrt(mean_squared_error(actual_prices, baseline_predictions))

print(f"Model MAE    : ${mae:.2f}")
print(f"Baseline MAE : ${baseline_mae:.2f}")
print(f"Model RMSE   : ${rmse:.2f}")
print(f"Baseline RMSE: ${baseline_rmse:.2f}")

## 13. Save Notebook Model Outputs

Run this cell only when you want the notebook training result to replace the app model files.

In [ ]:
SAVE_RESULTS = False

if SAVE_RESULTS:
    Path("models").mkdir(exist_ok=True)

    model.save(MODEL_PATH)
    joblib.dump(scaler, SCALER_PATH)

    history_data = {
        "history": history.history,
        "sequence_length": SEQUENCE_LENGTH,
        "train_ratio": TRAIN_RATIO,
        "batch_size": BATCH_SIZE,
        "epochs_requested": EPOCHS,
        "random_seed": RANDOM_SEED,
        "best_validation_loss": float(best_val_loss),
        "test_mae_dollars": float(mae),
        "test_rmse_dollars": float(rmse),
        "test_mape_percent": float(mape),
    }

    with open(HISTORY_PATH, "w") as file:
        json.dump(history_data, file, indent=4)

    print("Saved model, scaler, and history.")
else:
    print("SAVE_RESULTS is False, so app files were not overwritten.")

## 14. One Prediction Example

This predicts the next price after the latest 60 prices in the CSV file.

In [ ]:
last_60_prices = close_prices.tail(SEQUENCE_LENGTH).values.reshape(-1, 1)
last_60_scaled = scaler.transform(last_60_prices)
model_input = last_60_scaled.reshape(1, SEQUENCE_LENGTH, 1)

next_scaled_price = model.predict(model_input, verbose=0)
next_price = scaler.inverse_transform(next_scaled_price)[0][0]
current_price = close_prices.iloc[-1]
expected_change = next_price - current_price
expected_change_percent = (expected_change / current_price) * 100

print(f"Current price          : ${current_price:.2f}")
print(f"Predicted next price   : ${next_price:.2f}")
print(f"Expected change        : ${expected_change:.2f}")
print(f"Expected change percent: {expected_change_percent:.2f}%")

## Final Notes

- A small train/validation gap is normal.
- Severe overfitting means validation loss gets much worse while training loss keeps improving.
- This project avoids common leakage by fitting the scaler only on training data.
- Test sequences use previous training prices as context, not future prices.
- For stock prices, compare the model against the simple baseline because stock movement is noisy.